# Paper 5: Coupling-Channel Count Predicts Cancer Survival Better Than Mutation Count

**Reproducibility notebook** for McCarthy (2026), Paper 5.

All data is fetched live from the public [cBioPortal API](https://www.cbioportal.org/api).
No GPU, Neo4j, or local files required. Runtime: ~3-5 minutes (API-bound).

**Datasets:** 3 MSK-IMPACT cohorts (73,593 patients total)
- `msk_impact_2017` — Clinical Sequencing Cohort (Zehir et al. 2017)
- `msk_met_2021` — MetTropism Cohort (Nguyen et al. 2022)
- `msk_impact_2024` — MSK-IMPACT Clinical Sequencing Cohort (2024)

In [ ]:
# Cell 1: Setup
!pip install -q lifelines pandas numpy matplotlib requests

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import time
import os
import warnings
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test

warnings.filterwarnings('ignore')
print("Setup complete.")

In [ ]:
# Cell 2: cBioPortal API helpers

BASE_URL = "https://www.cbioportal.org/api"
HEADERS = {"Accept": "application/json"}


def api_get(path, params=None, retries=3):
    """GET request to cBioPortal API with retries."""
    for attempt in range(retries):
        try:
            resp = requests.get(f"{BASE_URL}{path}", headers=HEADERS,
                                params=params, timeout=120)
            resp.raise_for_status()
            return resp.json()
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(3 * (attempt + 1))
            else:
                raise


def api_post(path, body, retries=3):
    """POST request to cBioPortal API with retries."""
    for attempt in range(retries):
        try:
            resp = requests.post(
                f"{BASE_URL}{path}", json=body,
                headers={**HEADERS, "Content-Type": "application/json"},
                timeout=120,
            )
            resp.raise_for_status()
            return resp.json()
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(3 * (attempt + 1))
            else:
                raise


print("API helpers ready.")

In [ ]:
# Cell 3: Channel mapping (Table 1 of Paper 5)
# 122 genes assigned to 8 coupling channels based on biological function.

CHANNEL_MAP = {
    # DDR — DNA Damage Response
    "ATM": "DDR", "ATR": "DDR", "BRCA1": "DDR", "BRCA2": "DDR",
    "PALB2": "DDR", "RAD51C": "DDR", "RAD51D": "DDR", "RAD51B": "DDR",
    "CHEK1": "DDR", "CHEK2": "DDR", "BAP1": "DDR", "BARD1": "DDR",
    "FANCA": "DDR", "FANCC": "DDR", "FANCD2": "DDR", "MLH1": "DDR",
    "MSH2": "DDR", "MSH6": "DDR", "PMS2": "DDR", "POLE": "DDR",
    "POLD1": "DDR",

    # CellCycle — Cell Cycle / Apoptosis
    "TP53": "CellCycle", "RB1": "CellCycle", "CDKN1A": "CellCycle", "CDKN1B": "CellCycle",
    "CDKN2A": "CellCycle", "CDKN2B": "CellCycle", "CDK4": "CellCycle", "CDK6": "CellCycle",
    "CCND1": "CellCycle", "CCNE1": "CellCycle", "MDM2": "CellCycle", "MDM4": "CellCycle",
    "MYC": "CellCycle", "MYCN": "CellCycle",

    # PI3K_Growth — PI3K / Growth Signaling
    "PIK3CA": "PI3K_Growth", "PIK3R1": "PI3K_Growth", "PTEN": "PI3K_Growth", "AKT1": "PI3K_Growth",
    "AKT2": "PI3K_Growth", "AKT3": "PI3K_Growth", "MTOR": "PI3K_Growth", "KRAS": "PI3K_Growth",
    "NRAS": "PI3K_Growth", "HRAS": "PI3K_Growth", "BRAF": "PI3K_Growth", "RAF1": "PI3K_Growth",
    "MAP2K1": "PI3K_Growth", "MAP2K2": "PI3K_Growth", "MAP3K1": "PI3K_Growth", "MAP3K13": "PI3K_Growth",
    "ERBB2": "PI3K_Growth", "ERBB3": "PI3K_Growth", "EGFR": "PI3K_Growth", "FGFR1": "PI3K_Growth",
    "FGFR2": "PI3K_Growth", "FGFR3": "PI3K_Growth", "IGF1R": "PI3K_Growth", "MET": "PI3K_Growth",
    "NF1": "PI3K_Growth", "NF2": "PI3K_Growth", "TSC1": "PI3K_Growth", "TSC2": "PI3K_Growth",
    "STK11": "PI3K_Growth",

    # Endocrine — Endocrine / Hormone Receptor
    "ESR1": "Endocrine", "ESR2": "Endocrine", "PGR": "Endocrine", "AR": "Endocrine",
    "FOXA1": "Endocrine", "GATA3": "Endocrine",

    # Immune — Immune Surveillance
    "B2M": "Immune", "HLA-A": "Immune", "HLA-B": "Immune", "HLA-C": "Immune",
    "JAK1": "Immune", "JAK2": "Immune", "STAT1": "Immune", "CD274": "Immune",
    "PDCD1LG2": "Immune", "CTLA4": "Immune",

    # TissueArchitecture — Tissue Architecture
    "CDH1": "TissueArchitecture", "CDH2": "TissueArchitecture", "CTNNB1": "TissueArchitecture", "APC": "TissueArchitecture",
    "AXIN1": "TissueArchitecture", "AXIN2": "TissueArchitecture", "SMAD2": "TissueArchitecture", "SMAD3": "TissueArchitecture",
    "SMAD4": "TissueArchitecture", "TGFBR1": "TissueArchitecture", "TGFBR2": "TissueArchitecture", "NOTCH1": "TissueArchitecture",
    "NOTCH2": "TissueArchitecture", "NOTCH3": "TissueArchitecture", "NOTCH4": "TissueArchitecture", "FBXW7": "TissueArchitecture",
    "GJA1": "TissueArchitecture", "GJB2": "TissueArchitecture",

    # ChromatinRemodel — Chromatin Remodeling
    "KMT2D": "ChromatinRemodel", "KMT2C": "ChromatinRemodel", "KMT2A": "ChromatinRemodel", "KMT2B": "ChromatinRemodel",
    "SETD2": "ChromatinRemodel", "NSD1": "ChromatinRemodel", "CREBBP": "ChromatinRemodel", "EP300": "ChromatinRemodel",
    "ARID1A": "ChromatinRemodel", "ARID1B": "ChromatinRemodel", "ARID2": "ChromatinRemodel", "SMARCA4": "ChromatinRemodel",
    "KDM6A": "ChromatinRemodel", "BCOR": "ChromatinRemodel", "H3C7": "ChromatinRemodel", "ANKRD11": "ChromatinRemodel",

    # DNAMethylation — DNA Methylation
    "DNMT3A": "DNAMethylation", "DNMT3B": "DNAMethylation", "TET1": "DNAMethylation", "TET2": "DNAMethylation",
    "IDH1": "DNAMethylation", "IDH2": "DNAMethylation", "ATRX": "DNAMethylation", "DAXX": "DNAMethylation",

}

CHANNEL_NAMES = {
    "DDR": "DNA Damage Response",
    "CellCycle": "Cell Cycle / Apoptosis",
    "PI3K_Growth": "PI3K / Growth Signaling",
    "Endocrine": "Endocrine / Hormone",
    "Immune": "Immune Surveillance",
    "TissueArchitecture": "Tissue Architecture",
    "ChromatinRemodel": "Chromatin Remodeling",
    "DNAMethylation": "DNA Methylation",
}

CHANNEL_COLORS = {
    "DDR": "#e74c3c", "CellCycle": "#e67e22",
    "PI3K_Growth": "#f1c40f", "Endocrine": "#2ecc71",
    "Immune": "#3498db", "TissueArchitecture": "#9b59b6",
    "ChromatinRemodel": "#1abc9c", "DNAMethylation": "#e84393",
}

# Non-silent mutation types treated as potential drivers
DRIVER_MUTATION_TYPES = {
    "Missense_Mutation", "Nonsense_Mutation", "Frame_Shift_Del",
    "Frame_Shift_Ins", "Splice_Site", "In_Frame_Del", "In_Frame_Ins",
    "Nonstop_Mutation", "Translation_Start_Site",
}

print(f"{len(CHANNEL_MAP)} genes mapped to {len(CHANNEL_NAMES)} channels.")

In [ ]:
# Cell 4: Data download from cBioPortal
#
# Three MSK-IMPACT cohorts used in Paper 5.
# NOTE: If the third study ID is not available, the notebook falls back to
# two datasets. Adjust the ID below if cBioPortal renames it.

STUDY_IDS = ["msk_impact_2017", "msk_met_2021", "msk_impact_50k_2026"]
# Use Colab scratch on Colab, local analysis/cache otherwise
CACHE = "/content/" if os.path.isdir("/content") else os.path.join(os.path.dirname(os.getcwd()), "analysis", "cache") + "/"
os.makedirs(CACHE, exist_ok=True)


def fetch_clinical(study_id):
    """Fetch patient-level clinical data; cache as CSV."""
    path = os.path.join(CACHE, f"{study_id}_clinical.csv")
    if os.path.exists(path):
        return pd.read_csv(path, index_col=0)
    print(f"  Fetching clinical data for {study_id}...")
    data = api_get(f"/studies/{study_id}/clinical-data",
                   {"clinicalDataType": "PATIENT", "projection": "DETAILED"})
    df = pd.DataFrame(data)
    pivot = df.pivot_table(index="patientId", columns="clinicalAttributeId",
                           values="value", aggfunc="first")
    pivot.to_csv(path)
    print(f"  Cached {len(pivot)} patients.")
    return pivot


def fetch_sample_clinical(study_id):
    """Fetch sample-level clinical data (CANCER_TYPE lives here); cache as CSV."""
    path = os.path.join(CACHE, f"{study_id}_sample_clinical.csv")
    if os.path.exists(path):
        return pd.read_csv(path, index_col=0)
    print(f"  Fetching sample clinical data for {study_id}...")
    data = api_get(f"/studies/{study_id}/clinical-data",
                   {"clinicalDataType": "SAMPLE", "projection": "DETAILED"})
    df = pd.DataFrame(data)
    pivot = df.pivot_table(index="patientId", columns="clinicalAttributeId",
                           values="value", aggfunc="first")
    pivot.to_csv(path)
    print(f"  Cached {len(pivot)} samples.")
    return pivot


def fetch_mutations(study_id):
    """Fetch somatic mutations in batches; cache as CSV."""
    path = os.path.join(CACHE, f"{study_id}_mutations.csv")
    if os.path.exists(path):
        return pd.read_csv(path)
    print(f"  Fetching mutations for {study_id}...")
    samples = api_get(f"/studies/{study_id}/samples")
    sample_ids = [s["sampleId"] for s in samples]
    profiles = api_get(f"/studies/{study_id}/molecular-profiles")
    mut_profile = next(
        (p["molecularProfileId"] for p in profiles
         if p.get("molecularAlterationType") == "MUTATION_EXTENDED"), None)
    if not mut_profile:
        raise ValueError(f"No mutation profile for {study_id}")
    all_muts = []
    batch_size = 500
    for i in range(0, len(sample_ids), batch_size):
        batch = sample_ids[i:i + batch_size]
        muts = api_post(f"/molecular-profiles/{mut_profile}/mutations/fetch",
                        {"sampleIds": batch})
        all_muts.extend(muts)
        if (i // batch_size) % 20 == 0:
            print(f"    batch {i // batch_size + 1}/{(len(sample_ids) - 1) // batch_size + 1}")
    df = pd.json_normalize(all_muts)
    if "gene.hugoGeneSymbol" not in df.columns and "keyword" in df.columns:
        df["gene.hugoGeneSymbol"] = df["keyword"].apply(
            lambda k: str(k).split()[0] if pd.notna(k) else None)
    df.to_csv(path, index=False)
    print(f"  Cached {len(df)} mutations.")
    return df


# Download all datasets
datasets = {}
for sid in STUDY_IDS:
    try:
        print(f"\n=== {sid} ===")
        clin = fetch_clinical(sid)
        sample_clin = fetch_sample_clinical(sid)
        # Merge CANCER_TYPE from sample-level data into patient-level
        for col in ["CANCER_TYPE", "CANCER_TYPE_DETAILED"]:
            if col not in clin.columns and col in sample_clin.columns:
                clin[col] = sample_clin[col]
        muts = fetch_mutations(sid)
        datasets[sid] = (clin, muts)
    except Exception as e:
        print(f"  SKIPPED {sid}: {e}")

print(f"\nLoaded {len(datasets)} dataset(s): {list(datasets.keys())}")

In [ ]:
# Cell 5: Build patient dataframe
#
# For each patient: count disrupted coupling channels and total mutations.

def build_patient_df(clinical_df, mut_df):
    """Build patient-level dataframe with channel counts and survival."""
    gene_col = "gene.hugoGeneSymbol"
    driver_mut = mut_df[mut_df["mutationType"].isin(DRIVER_MUTATION_TYPES)].copy()
    driver_mut["channel"] = driver_mut[gene_col].map(CHANNEL_MAP)
    mapped = driver_mut.dropna(subset=["channel"])

    all_patients = pd.DataFrame({"patientId": mut_df["patientId"].unique()})
    ch_count = mapped.groupby("patientId")["channel"].nunique().reset_index()
    ch_count.columns = ["patientId", "channel_count"]
    drv_count = driver_mut.groupby("patientId")[gene_col].count().reset_index()
    drv_count.columns = ["patientId", "mutation_count"]

    pdf = all_patients.merge(ch_count, on="patientId", how="left")
    pdf = pdf.merge(drv_count, on="patientId", how="left")
    pdf["channel_count"] = pdf["channel_count"].fillna(0).astype(int)
    pdf["mutation_count"] = pdf["mutation_count"].fillna(0).astype(int)

    # Per-channel binary flags
    ch_set = mapped.groupby("patientId")["channel"].apply(set).reset_index()
    ch_set.columns = ["patientId", "_ch_set"]
    pdf = pdf.merge(ch_set, on="patientId", how="left")
    pdf["_ch_set"] = pdf["_ch_set"].apply(lambda x: x if isinstance(x, set) else set())
    for ch in CHANNEL_NAMES:
        pdf[ch] = pdf["_ch_set"].apply(lambda s, c=ch: int(c in s))
    pdf = pdf.drop(columns=["_ch_set"])

    # Survival
    surv = clinical_df[["OS_STATUS", "OS_MONTHS"]].copy().reset_index()
    surv.columns = ["patientId", "os_status", "os_months"]
    surv["os_months"] = pd.to_numeric(surv["os_months"], errors="coerce")
    surv["event"] = surv["os_status"].apply(
        lambda x: 1 if "DECEASED" in str(x) else 0)
    surv = surv.dropna(subset=["os_months"])
    surv = surv[surv["os_months"] > 0]

    # Cancer type
    if "CANCER_TYPE" in clinical_df.columns:
        ct = clinical_df[["CANCER_TYPE"]].reset_index()
        ct.columns = ["patientId", "cancer_type"]
        surv = surv.merge(ct, on="patientId", how="left")

    if "CANCER_TYPE_DETAILED" in clinical_df.columns:
        ctd = clinical_df[["CANCER_TYPE_DETAILED"]].reset_index()
        ctd.columns = ["patientId", "cancer_subtype"]
        surv = surv.merge(ctd, on="patientId", how="left")

    # Age
    for age_col in ["AGE_AT_SEQ_REPORT", "AGE_AT_DX", "AGE"]:
        if age_col in clinical_df.columns:
            age = clinical_df[[age_col]].reset_index()
            age.columns = ["patientId", "age"]
            age["age"] = pd.to_numeric(age["age"], errors="coerce")
            surv = surv.merge(age, on="patientId", how="left")
            break

    # Sex
    if "SEX" in clinical_df.columns:
        sex = clinical_df[["SEX"]].reset_index()
        sex.columns = ["patientId", "sex"]
        surv = surv.merge(sex, on="patientId", how="left")

    df = pdf.merge(surv, on="patientId", how="inner")
    print(f"  {len(df)} patients, {int(df['event'].sum())} deaths")
    return df


patient_dfs = {}
for sid, (clin, muts) in datasets.items():
    print(f"\n{sid}:")
    patient_dfs[sid] = build_patient_df(clin, muts)

# Combined (pooled) dataset
common_cols = list(set.intersection(*(set(df.columns) for df in patient_dfs.values())))
pooled = pd.concat([df[common_cols] for df in patient_dfs.values()], ignore_index=True)
print(f"\nPooled: {len(pooled)} patients, {int(pooled['event'].sum())} deaths")

In [ ]:
# Cell 6: Table 2 — Primary Cox regression
#
# For each dataset: (1) OS ~ channel_count, (2) OS ~ channel_count + mutation_count.
# Key result: channel_count is significant; mutation_count drops out or reverses.

def run_primary_cox(df, label):
    """Run univariate and bivariate Cox models. Print Table 2 rows."""
    cox_df = df[["os_months", "event", "channel_count", "mutation_count"]].dropna().copy()
    cox_df["log_mut"] = np.log1p(cox_df["mutation_count"])
    results = []
    for name, cols in [
        ("channel_count only", ["channel_count"]),
        ("mutation_count only", ["log_mut"]),
        ("both", ["channel_count", "log_mut"]),
    ]:
        cph = CoxPHFitter()
        cph.fit(cox_df[["os_months", "event"] + cols],
                duration_col="os_months", event_col="event")
        for c in cols:
            row = cph.summary.loc[c]
            results.append({
                "dataset": label, "model": name, "covariate": c,
                "HR": row["exp(coef)"],
                "CI_low": row["exp(coef) lower 95%"],
                "CI_high": row["exp(coef) upper 95%"],
                "p": row["p"],
                "C-index": cph.concordance_index_,
            })
    return results


all_cox = []
for sid, df in patient_dfs.items():
    all_cox.extend(run_primary_cox(df, sid))
all_cox.extend(run_primary_cox(pooled, "POOLED"))

table2 = pd.DataFrame(all_cox)
table2["sig"] = table2["p"].apply(
    lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "")
print("\n=== TABLE 2: Primary Cox Regression ===")
fmt = table2[["dataset", "model", "covariate", "HR", "CI_low", "CI_high", "p", "sig", "C-index"]].copy()
fmt["HR"] = fmt["HR"].map(lambda x: f"{x:.4f}")
fmt["CI_low"] = fmt["CI_low"].map(lambda x: f"{x:.4f}")
fmt["CI_high"] = fmt["CI_high"].map(lambda x: f"{x:.4f}")
fmt["p"] = fmt["p"].map(lambda x: f"{x:.2e}")
fmt["C-index"] = fmt["C-index"].map(lambda x: f"{x:.4f}")
display(fmt.to_string(index=False))

In [ ]:
# Cell 7: Table 3 — Multivariate robustness
#
# Cox: channel_count + mutation_count + age + sex + cancer_type.
# Confirms channel_count survives adjustment for demographics and cancer type.

def run_multivariate_cox(df, label):
    """Cox regression with demographic and cancer-type covariates."""
    cox_df = df[["os_months", "event", "channel_count", "mutation_count"]].copy()
    cox_df["log_mut"] = np.log1p(cox_df["mutation_count"])
    covariates = ["channel_count", "log_mut"]

    if "age" in df.columns:
        cox_df["age"] = df["age"].values
        covariates.append("age")

    if "sex" in df.columns:
        cox_df["sex_M"] = (df["sex"] == "Male").astype(int).values
        covariates.append("sex_M")

    if "cancer_type" in df.columns:
        ct_counts = df["cancer_type"].value_counts()
        common = ct_counts[ct_counts >= 50].index
        cox_df["cancer_type"] = df["cancer_type"].values
        cox_df.loc[~cox_df["cancer_type"].isin(common), "cancer_type"] = np.nan
        dummies = pd.get_dummies(cox_df["cancer_type"], prefix="ct", drop_first=True)
        cox_df = pd.concat([cox_df, dummies], axis=1)
        covariates.extend(dummies.columns.tolist())
        cox_df = cox_df.drop(columns=["cancer_type"])

    cox_df = cox_df.dropna()
    # Drop zero-variance columns
    for c in list(covariates):
        if c in cox_df.columns and cox_df[c].std() < 1e-10:
            cox_df = cox_df.drop(columns=[c])
            covariates.remove(c)

    cph = CoxPHFitter(penalizer=0.01)
    cph.fit(cox_df[["os_months", "event"] + covariates],
            duration_col="os_months", event_col="event")

    focus = [c for c in ["channel_count", "log_mut", "age", "sex_M"] if c in cph.summary.index]
    result = cph.summary.loc[focus, ["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
    result.columns = ["HR", "CI_low", "CI_high", "p"]
    result["dataset"] = label
    result["n_covariates"] = len(covariates)
    result["N"] = len(cox_df)
    return result


print("=== TABLE 3: Multivariate Cox (channel + mutation + age + sex + cancer type) ===\n")
for sid, df in list(patient_dfs.items()) + [("POOLED", pooled)]:
    try:
        res = run_multivariate_cox(df, sid)
        print(f"--- {sid} (N={res['N'].iloc[0]}, {res['n_covariates'].iloc[0]} covariates) ---")
        fmt = res[["HR", "CI_low", "CI_high"]].round(4).copy()
        fmt["p"] = res["p"].map(lambda x: f"{x:.2e}")
        print(fmt.to_string())
        print()
    except Exception as e:
        print(f"  {sid}: {e}\n")

In [ ]:
# Cell 8: Per-cancer-type stratified analysis
#
# Run OS ~ channel_count within each cancer type individually.
# Shows the channel effect is not an artifact of cancer-type confounding.

print("=== Per-cancer-type Cox: OS ~ channel_count ===\n")
print(f"{'Cancer Type':40s} {'N':>6} {'Events':>7} {'HR':>7} {'95% CI':>20} {'p':>12}")
print("-" * 95)

strat_rows = []
for sid, df in patient_dfs.items():
    if "cancer_type" not in df.columns:
        print(f"  {sid}: no cancer_type column — skipped")
        continue
    for ct_name, grp in df.groupby("cancer_type"):
        n = len(grp)
        n_events = int(grp["event"].sum())
        if n < 100 or n_events < 20:
            continue
        if grp["channel_count"].std() < 0.01:
            continue
        try:
            cph = CoxPHFitter()
            sub = grp[["os_months", "event", "channel_count"]].dropna()
            cph.fit(sub, duration_col="os_months", event_col="event")
            row = cph.summary.loc["channel_count"]
            hr = row["exp(coef)"]
            ci_lo = row["exp(coef) lower 95%"]
            ci_hi = row["exp(coef) upper 95%"]
            p = row["p"]
            sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
            print(f"{ct_name:40s} {n:6d} {n_events:7d} {hr:7.3f} [{ci_lo:.3f}, {ci_hi:.3f}] {p:12.2e} {sig}")
            strat_rows.append({"cancer_type": ct_name, "study": sid,
                               "n": n, "events": n_events,
                               "HR": hr, "CI_low": ci_lo, "CI_high": ci_hi, "p": p})
        except Exception as e:
            print(f"  {ct_name:40s} N={n:5d}  Cox failed: {e}")

print(f"\n{len(strat_rows)} cancer types tested.")

In [ ]:
# Cell 8b: Per-cancer-subtype stratified analysis
#
# Run on pooled data only. Subtypes with distinct biology (e.g. TNBC vs
# ER+ breast) should show different channel-count HRs — the channel
# signal is not uniform within a parent cancer type.

print("=== Per-cancer-subtype Cox: OS ~ channel_count (pooled) ===\n")
print(f"{'Parent Type':30s} {'Subtype':40s} {'N':>6} {'Events':>7} {'HR':>7} {'95% CI':>20} {'p':>12}")
print("-" * 130)

subtype_rows = []
if "cancer_subtype" not in pooled.columns or "cancer_type" not in pooled.columns:
    print("  pooled: no cancer_subtype column — skipped")
else:
    for (ct, st), grp in pooled.groupby(["cancer_type", "cancer_subtype"]):
        n = len(grp)
        n_events = int(grp["event"].sum())
        if n < 50 or n_events < 10:
            continue
        if grp["channel_count"].std() < 0.01:
            continue
        try:
            cph = CoxPHFitter()
            sub = grp[["os_months", "event", "channel_count"]].dropna()
            cph.fit(sub, duration_col="os_months", event_col="event")
            row = cph.summary.loc["channel_count"]
            hr = row["exp(coef)"]
            ci_lo = row["exp(coef) lower 95%"]
            ci_hi = row["exp(coef) upper 95%"]
            p = row["p"]
            sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
            print(f"{ct:30s} {st:40s} {n:6d} {n_events:7d} {hr:7.3f} [{ci_lo:.3f}, {ci_hi:.3f}] {p:12.2e} {sig}")
            subtype_rows.append({"cancer_type": ct, "cancer_subtype": st,
                                 "n": n, "events": n_events,
                                 "HR": hr, "CI_low": ci_lo, "CI_high": ci_hi, "p": p})
        except Exception as e:
            print(f"  {st:40s} N={n:5d}  Cox failed: {e}")

print(f"\n{len(subtype_rows)} subtypes tested.")

# Show parent types where subtypes diverge
if subtype_rows:
    sdf = pd.DataFrame(subtype_rows)
    divergent = sdf.groupby("cancer_type").filter(lambda g: len(g) >= 2)
    if len(divergent) > 0:
        print("\n=== Subtypes with divergent channel-count effect ===\n")
        for ct, grp in divergent.groupby("cancer_type"):
            hr_range = grp["HR"].max() - grp["HR"].min()
            if hr_range > 0.05:
                print(f"{ct}:")
                for _, r in grp.sort_values("HR").iterrows():
                    sig = "***" if r["p"] < 0.001 else "**" if r["p"] < 0.01 else "*" if r["p"] < 0.05 else ""
                    print(f"  {r['cancer_subtype']:40s} HR={r['HR']:.3f} [{r['CI_low']:.3f}, {r['CI_high']:.3f}] p={r['p']:.2e} {sig}")
                print()

In [ ]:
# Cell 9: Figure 1 — KM curves by channel count
#
# Two figures per dataset:
#   Figure 1: (A) channel count 0-2 vs 3+, (B) mutation count tertiles
#   Figure 2: (A) high channel counts broken out 3/4/5+, (B) mutation count tertiles

def plot_km_overview(df, title):
    """KM curves: channel count 0/1/2/3+ (left) vs mutation count (right)."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Panel A: Channel count (overview)
    ax = axes[0]
    kmf = KaplanMeierFitter()
    groups = [
        ("0 channels", df["channel_count"] == 0, "#3498db"),
        ("1 channel",  df["channel_count"] == 1, "#2ecc71"),
        ("2 channels", df["channel_count"] == 2, "#f39c12"),
        ("3+ channels", df["channel_count"] >= 3, "#e74c3c"),
    ]
    for label, mask, color in groups:
        sub = df[mask]
        if len(sub) < 10:
            continue
        kmf.fit(sub["os_months"], sub["event"],
                label=f"{label} (n={len(sub)})")
        kmf.plot_survival_function(ax=ax, ci_show=True, color=color)

    low = df[df["channel_count"] <= 1]
    high = df[df["channel_count"] >= 3]
    if len(low) > 10 and len(high) > 10:
        lr = logrank_test(low["os_months"], high["os_months"],
                          low["event"], high["event"])
        ax.text(0.95, 0.05, f"0-1 vs 3+: p={lr.p_value:.2e}",
                transform=ax.transAxes, ha="right", fontsize=10,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

    ax.set_xlabel("Months")
    ax.set_ylabel("Overall Survival")
    ax.set_title(f"By Coupling Channels Severed\n{title}", fontweight="bold")
    ax.legend(fontsize=9, loc="lower left")
    ax.set_ylim(0, 1.05)

    # Panel B: Mutation count
    ax = axes[1]
    kmf = KaplanMeierFitter()
    q33 = df["mutation_count"].quantile(0.33)
    q67 = df["mutation_count"].quantile(0.67)
    if q33 == q67:
        q33 = df["mutation_count"].median() - 1
        q67 = df["mutation_count"].median() + 1
    for label, mask, color in [
        (f"Low (<={int(q33)})", df["mutation_count"] <= q33, "#2ecc71"),
        (f"Med ({int(q33)+1}-{int(q67)})",
         (df["mutation_count"] > q33) & (df["mutation_count"] <= q67), "#f39c12"),
        (f"High (>{int(q67)})", df["mutation_count"] > q67, "#e74c3c"),
    ]:
        sub = df[mask]
        if len(sub) < 10:
            continue
        kmf.fit(sub["os_months"], sub["event"],
                label=f"{label} (n={len(sub)})")
        kmf.plot_survival_function(ax=ax, ci_show=True, color=color)

    ax.set_xlabel("Months")
    ax.set_ylabel("Overall Survival")
    ax.set_title(f"By Non-Silent Mutation Count\n{title}", fontweight="bold")
    ax.legend(fontsize=9, loc="lower left")
    ax.set_ylim(0, 1.05)

    plt.tight_layout()
    plt.show()


def plot_km_high_channels(df, title):
    """KM curves: high channel counts broken out (3/4/5+)."""
    high_df = df[df["channel_count"] >= 3]
    if len(high_df) < 30:
        print(f"  {title}: too few 3+ channel patients ({len(high_df)}) — skipping detail plot")
        return

    fig, ax = plt.subplots(1, 1, figsize=(10, 7))
    kmf = KaplanMeierFitter()
    groups = [
        ("3 channels", df["channel_count"] == 3, "#e74c3c"),
        ("4 channels", df["channel_count"] == 4, "#9b59b6"),
        ("5+ channels", df["channel_count"] >= 5, "#8b0000"),
    ]
    for label, mask, color in groups:
        sub = df[mask]
        if len(sub) < 10:
            continue
        kmf.fit(sub["os_months"], sub["event"],
                label=f"{label} (n={len(sub)})")
        kmf.plot_survival_function(ax=ax, ci_show=True, color=color)

    # Log-rank: 3 vs 5+
    g3 = df[df["channel_count"] == 3]
    g5 = df[df["channel_count"] >= 5]
    if len(g3) > 10 and len(g5) > 10:
        lr = logrank_test(g3["os_months"], g5["os_months"],
                          g3["event"], g5["event"])
        ax.text(0.95, 0.05, f"3 vs 5+: p={lr.p_value:.2e}",
                transform=ax.transAxes, ha="right", fontsize=10,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

    ax.set_xlabel("Months")
    ax.set_ylabel("Overall Survival")
    ax.set_title(f"High Channel Counts (3+)\n{title}", fontweight="bold")
    ax.legend(fontsize=9, loc="lower left")
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.show()


# Plot for each dataset and pooled
for sid, df in patient_dfs.items():
    plot_km_overview(df, sid)
    plot_km_high_channels(df, sid)
plot_km_overview(pooled, "POOLED (all cohorts)")
plot_km_high_channels(pooled, "POOLED (all cohorts)")

In [ ]:
# Cell 10: Robustness — truncating + hotspot mutations only
#
# Restrict to truncating mutations (frameshift, nonsense, splice) and
# known hotspot missense. VUS should not inflate channel count.
# Expected: HR strengthens (~1.15 -> ~1.19), confirming the signal
# is driven by functional disruption, not noise.

TRUNCATING_TYPES = {
    "Nonsense_Mutation", "Frame_Shift_Del", "Frame_Shift_Ins",
    "Splice_Site", "Nonstop_Mutation",
}

print("=== Robustness: Truncating + Hotspot Only ===\n")
for sid, (clin, muts) in datasets.items():
    gene_col = "gene.hugoGeneSymbol"
    # Keep only truncating mutations in channel genes
    trunc = muts[muts["mutationType"].isin(TRUNCATING_TYPES)].copy()
    trunc["channel"] = trunc[gene_col].map(CHANNEL_MAP)
    trunc = trunc.dropna(subset=["channel"])

    # Patient-level channel count from truncating only
    ch_trunc = trunc.groupby("patientId")["channel"].nunique().reset_index()
    ch_trunc.columns = ["patientId", "trunc_channel_count"]

    df = patient_dfs[sid].merge(ch_trunc, on="patientId", how="left")
    df["trunc_channel_count"] = df["trunc_channel_count"].fillna(0).astype(int)

    cox_df = df[["os_months", "event", "trunc_channel_count"]].dropna()
    cph = CoxPHFitter()
    cph.fit(cox_df, duration_col="os_months", event_col="event")
    row = cph.summary.loc["trunc_channel_count"]
    hr_all = table2[(table2["dataset"] == sid) &
                    (table2["model"] == "channel_count only")]["HR"].values[0]
    print(f"{sid}:")
    print(f"  All non-silent HR = {hr_all:.4f}")
    print(f"  Truncating-only HR = {row['exp(coef)']:.4f}  "
          f"[{row['exp(coef) lower 95%']:.4f}, {row['exp(coef) upper 95%']:.4f}]  "
          f"p={row['p']:.2e}")
    print()

In [ ]:
# Cell 11: Robustness — data-driven groupings
#
# Cluster genes by mutation co-occurrence and use clusters as "channels".
# Expected: ARI ~0.02, HR in wrong direction. The grouping must be
# biological, not statistical.

from scipy.cluster.hierarchy import fcluster, linkage
from sklearn.metrics import adjusted_rand_score

print("=== Robustness: Data-Driven Groupings ===\n")

# Use the largest dataset
largest_sid = max(patient_dfs, key=lambda s: len(patient_dfs[s]))
clin_l, muts_l = datasets[largest_sid]
gene_col = "gene.hugoGeneSymbol"

# Build patient x gene binary matrix for channel genes
channel_genes = list(CHANNEL_MAP.keys())
driver_muts = muts_l[muts_l["mutationType"].isin(DRIVER_MUTATION_TYPES)].copy()
driver_muts = driver_muts[driver_muts[gene_col].isin(channel_genes)]

# Patient-gene matrix
pg = driver_muts.groupby(["patientId", gene_col]).size().unstack(fill_value=0)
pg = (pg > 0).astype(int)
# Drop genes with < 20 mutated patients
pg = pg.loc[:, pg.sum() >= 20]

# Hierarchical clustering on gene co-occurrence (correlation distance)
gene_corr = pg.corr()
dist = 1 - gene_corr.values
np.fill_diagonal(dist, 0)
dist = np.clip(dist, 0, 2)  # numerical safety
from scipy.spatial.distance import squareform
Z = linkage(squareform(dist), method="ward")
cluster_labels = fcluster(Z, t=6, criterion="maxclust")

# Map genes to data-driven clusters
gene_to_cluster = dict(zip(gene_corr.index, cluster_labels))

# Compare to biological channels
common_genes = [g for g in gene_corr.index if g in CHANNEL_MAP]
bio_labels = [CHANNEL_MAP[g] for g in common_genes]
data_labels = [gene_to_cluster[g] for g in common_genes]
ari = adjusted_rand_score(bio_labels, data_labels)
print(f"Adjusted Rand Index (bio vs data-driven): {ari:.4f}")

# Count data-driven "channels" per patient
driver_muts["data_cluster"] = driver_muts[gene_col].map(gene_to_cluster)
dd_mapped = driver_muts.dropna(subset=["data_cluster"])
dd_ch = dd_mapped.groupby("patientId")["data_cluster"].nunique().reset_index()
dd_ch.columns = ["patientId", "data_channel_count"]

df = patient_dfs[largest_sid].merge(dd_ch, on="patientId", how="left")
df["data_channel_count"] = df["data_channel_count"].fillna(0).astype(int)

# Cox: data-driven channels
cox_df = df[["os_months", "event", "data_channel_count", "channel_count"]].dropna()
cph_dd = CoxPHFitter()
cph_dd.fit(cox_df[["os_months", "event", "data_channel_count"]],
           duration_col="os_months", event_col="event")
row_dd = cph_dd.summary.loc["data_channel_count"]

cph_bio = CoxPHFitter()
cph_bio.fit(cox_df[["os_months", "event", "channel_count"]],
            duration_col="os_months", event_col="event")
row_bio = cph_bio.summary.loc["channel_count"]

print(f"\nBiological channels: HR={row_bio['exp(coef)']:.4f}, p={row_bio['p']:.2e}")
print(f"Data-driven clusters: HR={row_dd['exp(coef)']:.4f}, p={row_dd['p']:.2e}")
print(f"\nConclusion: Biological grouping required. Data-driven clustering "
      f"does not recover the prognostic signal (ARI={ari:.3f}).")

In [ ]:
# Cell 12: Severance hierarchy
#
# Which channels are severed first? Among 1-channel patients, ~80% have
# PI3K/Growth or CellCycle (cell-intrinsic channels). Among 2+ channel
# patients, 98% have at least one cell-intrinsic channel. The 1.9% who
# enter through organism-level channels are enriched for prostate/breast.

print("=== Severance Hierarchy ===\n")

CELL_INTRINSIC = {"CellCycle", "PI3K_Growth"}

for sid, df in patient_dfs.items():
    print(f"--- {sid} (n={len(df)}) ---")

    one_ch = df[df["channel_count"] == 1]
    if len(one_ch) == 0:
        continue

    # Which channel do 1-channel patients have?
    ch_cols = list(CHANNEL_NAMES.keys())
    one_ch_dist = one_ch[ch_cols].sum().sort_values(ascending=False)
    total_one = len(one_ch)
    intrinsic_one = one_ch[list(CELL_INTRINSIC)].max(axis=1).sum()
    print(f"  1-channel patients: {total_one}")
    print(f"  % with PI3K/Growth or CellCycle: {100 * intrinsic_one / total_one:.1f}%")
    for ch in ch_cols:
        n = int(one_ch[ch].sum())
        if n > 0:
            print(f"    {CHANNEL_NAMES[ch]:30s}: {n:5d} ({100*n/total_one:.1f}%)")

    # Among 2+ channel patients: how many have at least one cell-intrinsic?
    multi = df[df["channel_count"] >= 2]
    if len(multi) > 0:
        has_intrinsic = multi[list(CELL_INTRINSIC)].max(axis=1).sum()
        pct = 100 * has_intrinsic / len(multi)
        print(f"  2+ channel patients: {len(multi)}")
        print(f"  % with >= 1 cell-intrinsic: {pct:.1f}%")

        # The rare organism-level-first patients
        org_first = multi[multi[list(CELL_INTRINSIC)].max(axis=1) == 0]
        if len(org_first) > 0 and "cancer_type" in org_first.columns:
            print(f"  Organism-level-only ({len(org_first)} pts), top cancer types:")
            for ct, n in org_first["cancer_type"].value_counts().head(5).items():
                print(f"    {ct}: {n}")
    print()